In [1]:
%pip -q install langchain langchain-groq

In [2]:
from langchain_core.tools import tool

catalogue = {
    "black and white printing": "₹2 per printed side",
    "colour printing": "₹10 per printed side",
    "spiral binding": "₹30 per copy"
}

@tool
def get_price(service: str) -> str:
    """Look up the unit price of black and white printing, colour printing, or spiral binding."""
    name = service.strip().lower()
    return catalogue.get(name, "This service has no listed price.")

@tool
def is_service_available(service: str) -> bool:
    """Check whether the Campus Print Shop offers a service in its catalogue."""
    name = service.strip().lower()
    return name in catalogue

# Test each tool directly
print("Colour printing:", get_price.invoke({"service": "colour printing"}))
print("Spiral binding offered:", is_service_available.invoke({"service": "spiral binding"}))
print("Lamination offered:", is_service_available.invoke({"service": "lamination"}))

Colour printing: ₹10 per printed side
Spiral binding offered: True
Lamination offered: False


In [3]:
from typing import Literal
from pydantic import BaseModel, Field

class PrintOrder(BaseModel):
    """Details extracted from a customer's print request."""

    customer_name: str | None = Field(
        default=None, description="Customer's name, if given."
    )
    pages_per_copy: int | None = Field(
        default=None, ge=1, description="Number of pages in one copy."
    )
    copies: int | None = Field(
        default=None, ge=1, description="Number of copies requested."
    )
    print_type: Literal["black and white", "colour", "mixed"] | None = Field(
        default=None, description="Use mixed if the order includes both types."
    )
    spiral_binding: bool | None = Field(
        default=None, description="True if requested, False if declined, None if unspecified."
    )

# Check the record using one sample order
sample = PrintOrder(
    customer_name="Poojitha",
    pages_per_copy=20,
    copies=2,
    spiral_binding=True
)

print(sample.model_dump_json(indent=2))

{
  "customer_name": "Poojitha",
  "pages_per_copy": 20,
  "copies": 2,
  "print_type": null,
  "spiral_binding": true
}


In [4]:
from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("GROQ_API_KEY"),
    temperature=0
)

extract_order = llm.with_structured_output(PrintOrder, method="function_calling")

instructions = """
Extract a print order from the customer's message.
Use only details the customer gives. Leave missing fields as null.
'b&w' and 'bw' mean black and white. 'clr' means colour.
If some pages are colour and others are black and white, use mixed.
For spiral_binding, use true if requested, false if declined,
and null if the customer says nothing about binding.
"""

messages = [
    "poojitha here.. 20 pages b&w, 2 copies pls, spiral bind both",
    "ravi here need 1 copy of my 8 pg clr project. no binding pls",
    "hey can u print 3 copies of my 12 page assignment?",
    "sneha here, 10 pages total: colour cover + 9 bw pages, one copy",
    "need spiral binding pls, will send page count later"
]

records = []

for number, message in enumerate(messages, start=1):
    record = extract_order.invoke([
        ("system", instructions),
        ("human", message)
    ])
    records.append(record)

    print(f"\nMessage {number}: {message}")
    print(record.model_dump_json(indent=2))


Message 1: poojitha here.. 20 pages b&w, 2 copies pls, spiral bind both
{
  "customer_name": "poojitha",
  "pages_per_copy": 20,
  "copies": 2,
  "print_type": "black and white",
  "spiral_binding": true
}

Message 2: ravi here need 1 copy of my 8 pg clr project. no binding pls
{
  "customer_name": "ravi",
  "pages_per_copy": 8,
  "copies": 1,
  "print_type": "colour",
  "spiral_binding": false
}

Message 3: hey can u print 3 copies of my 12 page assignment?
{
  "customer_name": null,
  "pages_per_copy": 12,
  "copies": 3,
  "print_type": null,
  "spiral_binding": null
}

Message 4: sneha here, 10 pages total: colour cover + 9 bw pages, one copy
{
  "customer_name": "sneha",
  "pages_per_copy": 10,
  "copies": 1,
  "print_type": "mixed",
  "spiral_binding": null
}

Message 5: need spiral binding pls, will send page count later
{
  "customer_name": null,
  "pages_per_copy": null,
  "copies": null,
  "print_type": null,
  "spiral_binding": true
}


In [5]:
from google.colab import auth
from google.auth import default
import gspread

auth.authenticate_user()
credentials, _ = default()
client = gspread.authorize(credentials)

orders_sheet = client.open("Campus Print Orders").worksheet("Orders")
print(orders_sheet.row_values(1))

['customer_name', 'pages_per_copy', 'copies', 'print_type', 'spiral_binding']


In [6]:
@tool
def log_print_order(order: PrintOrder) -> str:
    """Save one print order to the Orders tab in Google Sheets."""
    row = [
        order.customer_name,
        order.pages_per_copy,
        order.copies,
        order.print_type,
        order.spiral_binding
    ]
    orders_sheet.append_row(
        ["" if value is None else value for value in row],
        value_input_option="RAW"
    )
    return "Order saved to Google Sheets."

# Save the first order you extracted earlier
print(log_print_order.invoke({"order": records[0].model_dump()}))

Order saved to Google Sheets.


In [7]:
from langchain.agents import create_agent

agent_prompt = "You are the Campus Print Shop assistant. Use the shop tools to answer customer questions."
question = "How much does spiral binding cost per copy?"

agent = create_agent(
    model=llm,
    tools=[get_price, is_service_available],
    system_prompt=agent_prompt
)

original_result = agent.invoke({
    "messages": [{"role": "user", "content": question}]
})

for message in original_result["messages"]:
    message.pretty_print()

================================ Human Message =================================

How much does spiral binding cost per copy?
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_5eec89d6-7965-4389-b641-e81e50896530)
 Call ID: fc_5eec89d6-7965-4389-b641-e81e50896530
  Args:
    service: spiral binding
================================= Tool Message =================================
Name: get_price

₹30 per copy
================================== Ai Message ==================================

Spiral binding costs **₹30 per copy**.


In [8]:
@tool
def tool1(service: str) -> str:
    """does stuff"""
    return get_price.invoke({"service": service})

test_agent = create_agent(
    model=llm,
    tools=[tool1, is_service_available],
    system_prompt=agent_prompt
)

# Start a fresh chat with the same question from Step 8.
renamed_result = test_agent.invoke({
    "messages": [{"role": "user", "content": question}]
})

for message in renamed_result["messages"]:
    message.pretty_print()

================================ Human Message =================================

How much does spiral binding cost per copy?
================================== Ai Message ==================================
Tool Calls:
  is_service_available (fc_8fc868f7-77c2-4d15-862d-69b508dd2a94)
 Call ID: fc_8fc868f7-77c2-4d15-862d-69b508dd2a94
  Args:
    service: spiral binding
================================= Tool Message =================================
Name: is_service_available

true
================================== Ai Message ==================================

I’m happy to confirm that spiral binding is available in our catalogue.  
Could you let me know how many copies you’re looking to bind? The price per copy can vary slightly based on quantity, so I’ll be able to give you the exact rate once I have that detail.


In [9]:
from langchain_core.messages import AIMessage, ToolMessage

model_calls = 0
tool_names = []

for message in original_result["messages"]:
    message.pretty_print()

    if isinstance(message, AIMessage):
        model_calls += 1

    if isinstance(message, ToolMessage):
        tool_names.append(message.name)

print("\nQuestion:", question)
print("Model calls:", model_calls)
print("Tool calls:", len(tool_names))
print("Tools used:", ", ".join(tool_names) or "None")

================================ Human Message =================================

How much does spiral binding cost per copy?
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_5eec89d6-7965-4389-b641-e81e50896530)
 Call ID: fc_5eec89d6-7965-4389-b641-e81e50896530
  Args:
    service: spiral binding
================================= Tool Message =================================
Name: get_price

₹30 per copy
================================== Ai Message ==================================

Spiral binding costs **₹30 per copy**.

Question: How much does spiral binding cost per copy?
Model calls: 2
Tool calls: 1
Tools used: get_price


In [10]:
%%writefile campus_print_shop_facts.txt
1. Campus Print Shop offers black-and-white printing, colour printing, and spiral binding.
2. Black-and-white printing costs ₹2 per printed side.
3. Colour printing costs ₹10 per printed side.
4. Spiral binding costs ₹30 per bound copy.
5. A mixed document contains both colour and black-and-white pages. Each printed side is charged at its own rate.
6. Single-sided printing uses one sheet of paper for each printed page.
7. Double-sided printing fits up to two printed pages on one sheet.
8. A 21-page document printed double-sided needs 11 sheets per copy.
9. Switching to double-sided printing saves paper. The printing price stays the same when the number and type of printed sides stay the same.
10. Binding is optional. An unbound copy has no binding charge.
11. Two separately bound copies have a total binding charge of ₹60, in addition to printing costs.
12. The order record stores the customer's name, pages per copy, number of copies, print type, and spiral binding choice.
13. Pages per copy means the number of pages in one complete copy of the document.
14. A missing binding choice means the customer has not yet specified whether they want binding.
15. A price estimate for a mixed document needs the number of colour pages and the number of black-and-white pages.
16. A “what if” question gives a price preview and leaves the customer's saved order unchanged.
17. Only the shop owner can confirm a pickup time.
18. Lamination is outside the shop's current service catalogue.

Writing campus_print_shop_facts.txt
